In [1]:
import pandas as pd
import numpy as np
data = pd.read_csv('laptops_Dataset.csv')

In [2]:
print(data.isnull().sum())

Brand:                      56
Product name                 0
Processor                    0
Video graphics               1
Video graphics Memorey     353
RAM                          0
Hard drive                   0
Display                      4
Display Resolution          20
Display Refresh Rate       353
Operating System             7
Keyboard                   101
Battery                     30
Webcam                     141
Connections                 24
Dimensions                  47
Weight                      40
Colors                      63
Warranty                     0
Price                        0
Processor Generation       337
Processor Details           70
Product number             351
Power supply type          929
Laptop Color              1020
Finger Print               969
SUPPORT SSD M2            1003
Pointing device           1008
Optical drive             1026
Series :                  1032
dtype: int64


In [3]:
data.drop(columns = ['Series :','Optical drive','Pointing device','SUPPORT SSD M2','Finger Print','Laptop Color',
                     'Power supply type','Product number'],inplace=True)

In [4]:
data.rename(columns = {
    'Brand:' : 'Brand', 'Product name' : 'Product_Name', 'Processor Generation' : 'Processor_Generation',
    'Processor Details' : 'Processor_Details', 'Display Refresh Rate' : 'Refresh_Rate', 'Display Resolution' : 'Resolution',
    'Operating System' : 'Os', 'Video graphics' : 'Video_Graphics', 'Video graphics Memorey' : 'Video_Graphics_Memory',
    'Hard drive' : 'Storage' },inplace=True)
data['Brand'].dropna().unique()

array(['MSI', 'Acer', 'Dell', 'HP', 'Lenovo', 'ASUS', 'Razer',
       'Microsoft', 'Huawei', 'Infinix', 'dynabook'], dtype=object)

In [5]:
data['Series'] = data['Product_Name'].str.lower().str.extract(
    '(victus|rog|tuf|f15|vivobook|zenbook|vostro|a15)'
)
series_to_brand = {
    'victus': 'HP', 'omen': 'HP', 'rog': 'ASUS', 'tuf': 'ASUS', 'vivobook': 'ASUS', 'zenbook': 'ASUS', 'legion': 'LENOVO',
    'ideapad': 'LENOVO', 'vostro': 'DELL', 'inspiron': 'DELL', 'a15': 'ACER','f15': 'ASUS' }

data['Brand'] = data['Brand'].fillna(data['Series'].map(series_to_brand))

In [6]:
data = data.copy()

data['Processor'] = data['Processor'].str.lower()
data['Processor'] = data['Processor'].str.replace(r'[^a-z0-9\- ]', '', regex=True)

data['CPU_Brand'] = data['Processor'].str.extract(r'(intel|amd|snapdragon)')

data['CPU_Series'] = (
    data['Processor']
    .str.lower()
    .str.extract(
        r'(i3|i5|i7|i9|celeron|amd\s?3020e|ryzen\s?[3579]|rayzen\s?[3579]|'
        r'ultra\s?[579]|core\s?[579]|ai\s?[79]|snapdragon\s?x\s?(?:elite|plus|\d+)?)'
    )[0]
)
data['CPU_Series'] = data['CPU_Series'].replace({
    r'snapdragon.*': 'snapdragon',
    r'rayzen': 'ryzen',
    r'core\s?5': 'i5',
    r'core\s?7': 'i7',
    r'core\s?9': 'i9'
}, regex=True)
data['CPU_Type'] = data['Processor'].str.extract(
    r'(hx|hs|h|u|p|g1|g7|g4)'
)

In [7]:
data = data.dropna(subset=['Brand'])
data.drop(columns = 
          ['Series','Video_Graphics','Video_Graphics_Memory','Display','Processor_Details','Processor_Generation','Keyboard'
           ,'Colors','Battery','Webcam','Connections','Dimensions','Resolution'],inplace = True)

In [8]:
data.isnull().sum()

Brand             0
Product_Name      0
Processor         0
RAM               0
Storage           0
Refresh_Rate    340
Os                7
Weight           38
Warranty          0
Price             0
CPU_Brand         3
CPU_Series        1
CPU_Type         12
dtype: int64

In [9]:
data = data.copy()

data['Warranty_Years'] = (
    data['Warranty']
    .astype(str)
    .str.lower()
    .replace({'سنه': '1 year', 'year': '1 year'})  # handle Arabic + "YEAR"
    .str.extract(r'(\d+)')                         # extract digits
    .astype(int)
    .fillna(1)                                     # fallback
)

data.drop(columns=['Warranty'], inplace=True)

data['Refresh_Rate'] = np.where(
    data['Product_Name'].str.lower().str.contains('gaming|rog|tuf|victus', na=False),
    144,60
)

In [10]:
data['CPU_Brand'] = data['CPU_Brand'].fillna('intel')

data['CPU_Series'] = data['CPU_Series'].fillna('i7')

data['CPU_Type'] = data['CPU_Type'].fillna(data.groupby(['CPU_Series'])['CPU_Type'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan))

data['CPU_Type'] = data['CPU_Type'].fillna(data['CPU_Type'].mode()[0])

In [11]:
data['Weight'] = data['Weight'].fillna(data.groupby(['CPU_Type','CPU_Series','CPU_Brand'])['Weight'].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan))

data['Storage'] = data['Storage'].str.extract(r'(512|256|1|2)').fillna(1).astype(int)

In [12]:
data = data.copy()

data['RAM_GB'] = (
    data['RAM'].astype(str).str.extract(r'(4|8|16|32)').astype(int).fillna(1)   # fallback
)
data.drop(columns=['RAM'], inplace=True)

In [13]:
data['OS'] = (
    data['Os'].astype(str).str.lower().str.extract(r'(10|11|dos)').astype(str).fillna(1)
)
data['OS'] = data['OS'].replace({
    'nan' : 'Windows 11',
    '11'  : 'Windows 11',
    '10'  : 'Windows 10'    
}).fillna(1)
data.drop(columns=['Os'],inplace=True)

In [14]:
data.isnull().sum()

Brand             0
Product_Name      0
Processor         0
Storage           0
Refresh_Rate      0
Weight            0
Price             0
CPU_Brand         0
CPU_Series        0
CPU_Type          0
Warranty_Years    0
RAM_GB            0
OS                0
dtype: int64

In [15]:
def map_call(cols,ext=''):
    for i in cols:
        print(i+ext,' :',end = ' ')
        print(data[i+ext].dropna().unique())

map_call([
    "Brand", "Storage", "CPU_Brand",
    "CPU_Series", "CPU_Type", "OS"
])

Brand  : ['MSI' 'Acer' 'Dell' 'HP' 'Lenovo' 'ASUS' 'Razer' 'ACER' 'Microsoft'
 'DELL' 'Huawei' 'Infinix' 'dynabook']
Storage  : [  2   1 512 256]
CPU_Brand  : ['intel' 'amd' 'snapdragon']
CPU_Series  : ['ultra 9' 'i9' 'i5' 'i7' 'ai 9' 'ai 7' 'ryzen 9' 'ryzen 7' 'ryzen 5'
 'ultra 7' 'i3' 'snapdragon' 'ultra 5' 'ryzen 3' 'celeron' 'amd 3020e']
CPU_Type  : ['u' 'hx' 'h' 'hs' 'p' 'g7' 'g1' 'g4']
OS  : ['Windows 11' 'dos' 'Windows 10']


In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

columns_to_encode = [
    "Brand", "CPU_Brand",
    "CPU_Series", "CPU_Type", "OS"
]

for col in columns_to_encode:
    data[col+'_enc'] = le.fit_transform(data[col])
map_call(columns_to_encode,ext='_enc')

Brand_enc  : [ 9  2  4  5  8  1 11  0 10  3  6  7 12]
CPU_Brand_enc  : [1 0 2]
CPU_Series_enc  : [15  7  5  6  1  0 11 10  9 14  4 12 13  8  3  2]
CPU_Type_enc  : [7 5 3 4 6 2 0 1]
OS_enc  : [1 2 0]


In [17]:
data.drop(columns =['Processor','Product_Name'],inplace=True)

In [33]:
data.to_csv('laptop_preprocessed.csv',index=False)

In [18]:
from scipy.stats import zscore

z_scores = np.abs(zscore(data.select_dtypes(include=np.number)))
# Threshold
threshold = 3
# Remove outliers
data = data[(z_scores < threshold).all(axis=1)]
print("Shape after outlier treatment:", data.shape)

Shape after outlier treatment: (967, 16)


In [19]:
from sklearn.model_selection import train_test_split,KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error

X = data.select_dtypes(include = np.number)
X = X.drop(columns=['Price'])
y = data['Price']
model = LinearRegression()

In [20]:
X_train, X_temp, y_train, y_temp = train_test_split(
X, y, test_size=0.3, random_state=42
)
# Second split: Validation + Test
X_val, X_test, y_val, y_test = train_test_split(
X_temp, y_temp, test_size=0.5, random_state=42
)
print("Training set size:", X_train.shape)
print("Validation set size:", X_val.shape)
print("Test set size:", X_test.shape)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

results = {
"Dataset": ["Training", "Validation", "Test"],
"MSE": [mean_squared_error(y_train, y_train_pred), mean_squared_error(y_val, y_val_pred), mean_squared_error(y_test, y_test_pred)],
"R2 Score": [r2_score(y_train, y_train_pred), r2_score(y_val, y_val_pred), r2_score(y_test, y_test_pred) ]
}
results_df = pd.DataFrame(results)
print(results_df)

Training set size: (676, 9)
Validation set size: (145, 9)
Test set size: (146, 9)
      Dataset           MSE  R2 Score
0    Training  3.545897e+08  0.574571
1  Validation  4.215704e+08  0.485274
2        Test  4.408756e+08  0.390500


In [21]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mse_scores = -cross_val_score( model, X, y, cv=kfold, scoring="neg_mean_squared_error" )

r2_scores = cross_val_score( model, X, y, cv=kfold, scoring="r2" )

print("MSE for each fold:", mse_scores)
print("Average MSE:", mse_scores.mean())
print("R2 for each fold:", r2_scores)
print("Average R2:", r2_scores.mean())

MSE for each fold: [4.82952676e+08 2.98723736e+08 4.16593857e+08 4.17870020e+08
 3.07135060e+08]
Average MSE: 384655069.6527084
R2 for each fold: [0.43239914 0.55007097 0.5742671  0.45992068 0.60798056]
Average R2: 0.5249276897439634


In [22]:
from IPython.display import Markdown
def performance(X,message):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    display(Markdown(f'***{message}***'))
    display(Markdown(f"**MSE :** {mean_squared_error(y_test,y_pred)}"))
    display(Markdown(f"**R2 Score :** {r2_score(y_test,y_pred)}"))

In [23]:
correlation = X.corrwith(y)
print(correlation.sort_values(ascending=False))
# Get column names where any correlation > 0.5
X_corr = correlation[correlation > 0.2].index

print('Correlation Features :',X_corr)
performance(X[X_corr],message='After Correlation')

RAM_GB            0.680374
CPU_Series_enc    0.297202
Refresh_Rate      0.158131
Warranty_Years    0.126737
CPU_Type_enc      0.121933
CPU_Brand_enc     0.008820
Brand_enc        -0.056861
OS_enc           -0.281983
Storage          -0.451995
dtype: float64
Correlation Features : Index(['RAM_GB', 'CPU_Series_enc'], dtype='object')


***After Correlation***

**MSE :** 493199905.52578324

**R2 Score :** 0.42035585272337495

In [24]:
rfe = RFE(model, n_features_to_select=4)
X_wrapper = rfe.fit_transform(X, y)
selected_features_rfe = X.columns[rfe.support_]
print("\nWrapper Selected Features:",selected_features_rfe)
performance(X[selected_features_rfe],'Wrapper Method(RFE)')


Wrapper Selected Features: Index(['RAM_GB', 'CPU_Brand_enc', 'CPU_Type_enc', 'OS_enc'], dtype='object')


***Wrapper Method(RFE)***

**MSE :** 477338507.8497946

**R2 Score :** 0.4389973127632192

In [25]:
random_forest_regression = RandomForestRegressor()
random_forest_regression.fit(X, y)

feature_importance = pd.Series(random_forest_regression.feature_importances_, 
                               index=X.columns)
top_features = feature_importance.sort_values(ascending=False).head(5)
print("Top Features :\n",top_features)
X_rfr = top_features.index
performance(X[X_rfr],'Embedded method ( RandomForestRegressor )')

Top Features :
 RAM_GB            0.515913
CPU_Series_enc    0.171779
CPU_Type_enc      0.086203
Brand_enc         0.077534
Storage           0.059633
dtype: float64


***Embedded method ( RandomForestRegressor )***

**MSE :** 479576558.4523101

**R2 Score :** 0.436366994065826

In [26]:
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X)
print(X_train_standard[:5])
performance(X_train_standard,'Scalar Standard')

[[-1.29477571 -0.59685948  1.73564055  0.34901465  1.29444823  0.47614113
   0.04367167  0.40562163 -0.58449157]
 [ 0.84185371 -0.59685948 -0.57615616 -0.76300238 -1.06159373  0.47614113
  -0.68444051 -0.65304535 -0.58449157]
 [ 0.84185371 -0.59685948 -0.57615616  0.34901465 -0.38843889  0.47614113
  -0.32038442 -0.65304535 -0.58449157]
 [-1.29477571 -0.59685948 -0.57615616  0.34901465 -0.38843889  0.47614113
   0.04367167 -0.65304535  1.04433653]
 [-1.29477571 -0.59685948 -0.57615616  2.57304872 -0.38843889  0.47614113
  -0.32038442 -0.65304535  1.04433653]]


***Scalar Standard***

**MSE :** 482952676.28017247

**R2 Score :** 0.4323991365753619

In [27]:
# Min-Max Scaling
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X)

print(X_train_minmax[:5])
performance(X_train_minmax,'MinMax Method')

[[0.         0.         1.         0.42857143 0.75       0.5
  0.46666667 0.71428571 0.5       ]
 [1.         0.         0.         0.14285714 0.16666667 0.5
  0.33333333 0.42857143 0.5       ]
 [1.         0.         0.         0.42857143 0.33333333 0.5
  0.4        0.42857143 0.5       ]
 [0.         0.         0.         0.42857143 0.33333333 0.5
  0.46666667 0.42857143 1.        ]
 [0.         0.         0.         1.         0.33333333 0.5
  0.4        0.42857143 1.        ]]


***MinMax Method***

**MSE :** 482952676.28017277

**R2 Score :** 0.43239913657536155